In [3]:
import io
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import plotly.express as px
warnings.filterwarnings('ignore')

%matplotlib inline

import polars as pl
from skimpy import skim
from summarytools import dfSummary
#summarytools doesnt work with polars library, it works with pandas tho

#%load_ext cudf.pandas
#supercharges workflow with GPU acceleration using cudf.pandas
#Im having pip install issues with cudf so commenting it out for now

In [12]:
#from pathlib import Path
#import pandas as pd
#import tarfile
#import urllib.request

#def load_fifa_data():
#    tarball_path = Path(r'C:\Users\admin\OneDrive\Documents\GitHub\Plotting\fifa_dataset_cleaned')
#    if not tarball_path.is_file():
#        Path("datasets").mkdir(parents=True, exist_ok=True)
#        url = "https://github.com/ageron/data/raw/main/housing.tgz"
#        urllib.request.urlretrieve(url, tarball_path)
#        with tarfile.open(tarball_path) as housing_tarball:
#            housing_tarball.extractall(path="datasets")
#    return pd.read_csv(Path("datasets/housing/fifa_training_2017_2022.csv"))

#fifa = load_fifa_data()

fifa = pd.read_csv(r'C:\Users\admin\OneDrive\Documents\GitHub\Plotting\fifa_dataset_cleaned\fifa_training_2017_2022.csv')
fifa.head(5)

,ID,Name,Age,Nationality,Overall,Potential,Club,Value,Wage,Special,...,Contract_Years_Remaining,Position_Group,Attacking_Composite,Passing_Composite,Defending_Composite,Physical_Composite,Technical_Composite,Pace_Composite,Int_Rep_Category,Market_Tier
0,176580,L. Suárez,29,Uruguay,92,92,FC Barcelona,83000000.0,525000.0,2291,...,4.0,Forward,89.000000,77.000000,37.666667,79.333333,87.666667,82.5,Worldwide,Superstar
1,178518,R. Nainggolan,28,Belgium,86,86,Roma,37500000.0,130000.0,2290,...,4.0,Other,82.000000,80.333333,83.666667,82.666667,79.333333,79.5,National,Star
2,181872,A. Vidal,29,Chile,87,87,FC Bayern München,41500000.0,180000.0,2285,...,2.0,Other,82.000000,82.000000,83.000000,84.666667,78.333333,75.5,Continental,Star
3,197445,D. Alaba,24,Austria,86,89,FC Bayern München,41500000.0,140000.0,2279,...,4.0,Defender,76.333333,80.333333,82.333333,79.333333,80.000000,86.0,Continental,Star
4,195864,P. Pogba,23,France,88,94,Manchester United,71500000.0,225000.0,2271,...,4.0,Other,82.666667,86.666667,71.333333,89.000000,87.666667,77.0,Continental,Superstar


In [13]:
fifa.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 102496 entries, 0 to 102495
Data columns (total 80 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   ID                        102496 non-null  int64  
 1   Name                      102496 non-null  object 
 2   Age                       102496 non-null  int64  
 3   Nationality               102496 non-null  object 
 4   Overall                   102496 non-null  int64  
 5   Potential                 102496 non-null  int64  
 6   Club                      102496 non-null  object 
 7   Value                     102496 non-null  float64
 8   Wage                      102496 non-null  float64
 9   Special                   102496 non-null  int64  
 10  Preferred Foot            102496 non-null  object 
 11  International Reputation  102496 non-null  float64
 12  Weak Foot                 102496 non-null  float64
 13  Skill Moves               102496 non-null  f

In [14]:
fifa.describe()

,ID,Age,Overall,Potential,Value,Wage,Special,International Reputation,Weak Foot,Skill Moves,...,Potential_Gap,Potential_Ratio,Contract_Year_Clean,Contract_Years_Remaining,Attacking_Composite,Passing_Composite,Defending_Composite,Physical_Composite,Technical_Composite,Pace_Composite
count,102496.000000,102496.000000,102496.000000,102496.000000,1.024960e+05,102496.000000,102496.000000,102496.000000,102496.000000,102496.000000,...,102496.000000,102496.000000,93832.000000,102496.000000,102496.000000,102496.000000,102496.000000,102496.000000,102496.000000,102496.000000
mean,216320.029260,24.745717,66.546763,71.852980,2.610242e+06,10763.595165,1612.502703,1.129429,2.962857,2.390727,...,5.306217,0.927477,2020.847547,1.368336,50.684150,55.702330,25.163652,64.541585,54.707956,65.261430
std,33472.202393,4.794615,6.968273,5.996143,6.000617e+06,21938.538108,269.118980,0.420776,0.666595,0.768737,...,5.595791,0.075735,1.873964,1.339247,17.094280,13.394603,27.743387,9.805115,17.012237,14.408945
min,2.000000,15.000000,42.000000,45.000000,0.000000e+00,250.000000,718.000000,1.000000,1.000000,1.000000,...,0.000000,0.653333,2011.000000,0.000000,4.666667,9.000000,0.000000,19.000000,6.666667,11.500000
25%,201451.000000,21.000000,62.000000,68.000000,3.750000e+05,2000.000000,1472.000000,1.000000,3.000000,2.000000,...,0.000000,0.875000,2020.000000,0.000000,38.000000,48.333333,0.000000,58.333333,48.000000,58.500000
50%,223989.000000,24.000000,67.000000,72.000000,8.250000e+05,4000.000000,1648.000000,1.000000,3.000000,2.000000,...,4.000000,0.947368,2021.000000,1.000000,54.333333,58.000000,14.666667,65.666667,58.333333,67.500000
75%,238449.250000,28.000000,71.000000,76.000000,2.300000e+06,10000.000000,1799.000000,1.000000,3.000000,3.000000,...,9.000000,1.000000,2022.000000,2.000000,64.000000,65.333333,55.333333,71.666667,66.333333,75.000000
max,264704.000000,54.000000,94.000000,95.000000,1.940000e+08,575000.000000,2349.000000,5.000000,5.000000,5.000000,...,26.000000,1.000000,2031.000000,9.000000,94.000000,93.666667,92.333333,91.666667,95.333333,97.000000


In [15]:
corr_matrix = fifa.corr(numeric_only=True)

In [19]:
corr_matrix["Wage_Clean"].sort_values(ascending=True)

Wage                       -0.215855
Value                      -0.188779
Value_Clean                -0.188779
International Reputation   -0.175428
Potential_Gap              -0.114308
                              ...   
Physical_Composite          0.051725
Age                         0.093761
Defending_Composite         0.102312
Potential_Ratio             0.125099
Wage_Clean                  1.000000
Name: Wage_Clean, Length: 62, dtype: float64